# Notebook 1 — Structure File I/O

## 1. Understand the POSCAR Format Manually
- read raw text
- parse scale
- parse lattice
- parse elements/counts
- parse coordinate mode
- calculate cell volume
- store parsed information in Python structures

## 2. Parse the Same Structure with pymatgen
- POSCAR → Structure object
- inspect lattice
- composition
- sites
- coordinates
- volume

## 3. Compare Manual Parsing and Library Parsing
- what information did we extract manually?
- what does pymatgen provide automatically?
- why scientific libraries are useful

## 4. Write Structure Objects Back to Files
- Structure → POSCAR

## 5. Summary

### 1. Understand the POSCAR Format Manually

In [1]:
from pathlib import Path
import numpy as np
import json

In [12]:
# find the path to the POSCAR file
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data_root = (
    project_root
    / "data"
    / "raw"
    / "mgo_defect_thermodynamics_dataset"
)

poscar_path = (
    data_root
    / "02_bulk_relaxed"
    / "POSCAR"
)

print("Current directory:", Path.cwd())
print("POSCAR path:", poscar_path)
print("File exists:", poscar_path.exists())

Current directory: /Users/yu/Desktop/defect_dynamic/notebooks
POSCAR path: /Users/yu/Desktop/defect_dynamic/data/raw/mgo_defect_thermodynamics_dataset/02_bulk_relaxed/POSCAR
File exists: True


In [13]:
# Parse the title and scaling factor
title = lines[0].strip()
scale = float(lines[1].strip())

print("title:", title)
print("scale:", scale)
print(type(title))
print(type(scale))

# Parse the lattice vectors
lattice_rows = lines[2:5]
lattice_data = []

for row in lattice_rows:
    row_data = [float(x) for x in row.split()]
    lattice_data.append(row_data)

lattice = np.array(lattice_data) * scale

print(lattice)
print(lattice.shape)

# Parse elements and their counts
elements = (lines[5].split()) #get a list of elements from the 6th line of the POSCAR file
counts = [int(value) for value in lines[6].split()]
natoms = sum(int(count) for count in counts)

print("elements:", elements)
print("counts:", counts)
print("natoms:", natoms)

assert elements == ["Mg", "O"]
assert counts == [32, 32]
assert natoms == 64

#Parse coordinate mode
coordinate_mode = lines[7].strip()

print("coordinate mode:", coordinate_mode)
assert coordinate_mode.lower() == "direct"

# Calculate volume of the unit cell
volume_A3 = np.linalg.det(lattice)
volume = np.abs(volume_A3) 
print("Volume:", volume, "Å³")

title: MgO 64-atom initial supercell
scale: 1.0
<class 'str'>
<class 'float'>
[[8.424 0.    0.   ]
 [0.    8.424 0.   ]
 [0.    0.    8.424]]
(3, 3)
elements: ['Mg', 'O']
counts: [32, 32]
natoms: 64
coordinate mode: Direct
Volume: 597.7988490240002 Å³


In [14]:
# Parse into a dictionary
header = {
    "title": title,
    "scale": scale,
    "lattice": lattice,
    "elements": elements,
    "counts": counts,
    "natoms": natoms,
    "coordinate_mode": coordinate_mode,
    "volume": volume
}

for key, value in header.items():
    print(key, ":", value)

title : MgO 64-atom initial supercell
scale : 1.0
lattice : [[8.424 0.    0.   ]
 [0.    8.424 0.   ]
 [0.    0.    8.424]]
elements : ['Mg', 'O']
counts : [32, 32]
natoms : 64
coordinate_mode : Direct
volume : 597.7988490240002


In [15]:
# Function
def parse_poscar_header(path: Path) -> dict:
    # TODO：splitlines
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()

    # TODO：parse title and scale
    title = lines[0].strip()
    scale = float(lines[1].strip())

    # TODO：parse lattice
    lattice_rows = lines[2:5]
    lattice_data = []
    for row in lattice_rows:
        row_data = [float(x) for x in row.split()]
        lattice_data.append(row_data)
    lattice = np.array(lattice_data) * scale

    # TODO：parse elements 和 counts
    elements = (lines[5].split()) #get a list of elements from the 6th line of the POSCAR file
    counts = [int(value) for value in lines[6].split()]

    # TODO：parse natoms
    natoms = sum(int(count) for count in counts)

    # TODO：parse coordinate mode
    coordinate_mode = lines[7].strip()

    # TODO：calculate volume
    volume_A3 = np.abs(np.linalg.det(lattice))

    # TODO：return a dictionary
    header = {
    "title": title,
    "scale": scale,
    "lattice": lattice,
    "elements": elements,
    "counts": counts,
    "natoms": natoms,
    "coordinate_mode": coordinate_mode,
    "volume_A3": volume_A3
    }   
    return header

In [16]:
#Test the function
bulk_header = parse_poscar_header(poscar_path)
print(bulk_header)
print(bulk_header["elements"])
print(bulk_header["counts"])
print(bulk_header["natoms"])
print(bulk_header["volume_A3"])

{'title': 'MgO 64-atom initial supercell', 'scale': 1.0, 'lattice': array([[8.424, 0.   , 0.   ],
       [0.   , 8.424, 0.   ],
       [0.   , 0.   , 8.424]]), 'elements': ['Mg', 'O'], 'counts': [32, 32], 'natoms': 64, 'coordinate_mode': 'Direct', 'volume_A3': np.float64(597.7988490240002)}
['Mg', 'O']
[32, 32]
64
597.7988490240002


## 2. Parse the Same Structure with pymatgen
- POSCAR → Structure object
- inspect lattice
- composition
- sites
- coordinates
- volume

In [3]:
from pathlib import Path
from pymatgen.core import Structure

In [4]:
#Find the path of POSCAR
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data_root = (
    project_root
    / "data"
    / "raw"
    / "mgo_defect_thermodynamics_dataset"
)

bulk_relaxation_dir = (
    data_root
    / "02_bulk_relaxed"
)

initial_structure_path = (
    bulk_relaxation_dir
    / "POSCAR"
)

# Test
print("Initial structure:", initial_structure_path)
print("Initial file exists:", initial_structure_path.exists())

Initial structure: /Users/yu/Desktop/defect_dynamic/data/raw/mgo_defect_thermodynamics_dataset/02_bulk_relaxed/POSCAR
Initial file exists: True


In [5]:
#Create an object from the POSCAR
initial_bulk = Structure.from_file(initial_structure_path)
print("Initial object type:", type(initial_bulk))

Initial object type: <class 'pymatgen.core.structure.Structure'>


In [17]:
##Get some basic information about the structure
print("Initial composition:", initial_bulk.composition)
print("Initial number of sites:", initial_bulk.num_sites)
print("Initial lattice lengths:", initial_bulk.lattice.abc)
print("Initial lattice angles:", initial_bulk.lattice.angles)
print("Initial volume:", initial_bulk.volume)
print("Initial space group:", initial_bulk.get_space_group_info())


Initial composition: Mg32 O32
Initial number of sites: 64
Initial lattice lengths: (8.424, 8.424, 8.424)
Initial lattice angles: (90.0, 90.0, 90.0)
Initial volume: 597.7988490239999
Initial space group: ('Fm-3m', 225)


## 3. Compare Manual Parsing and Library Parsing

### What information did we extract manually?

By reading the POSCAR as plain text, we manually extracted the main structural information:

- structure title
- scaling factor
- lattice vectors
- atomic species
- number of atoms for each species
- coordinate mode
- total number of atoms
- unit-cell volume

This helped reveal how the information is actually stored inside a POSCAR file and how raw text can be converted into Python data structures such as lists, NumPy arrays, and dictionaries.

### What does pymatgen provide automatically?

Using `pymatgen`, the same file can be converted directly into a `Structure` object.

The object already contains structured information such as:

- lattice
- composition
- atomic sites
- fractional and Cartesian coordinates
- unit-cell volume
- species information

This removes the need to manually interpret the file format each time.

### Why use scientific libraries?

Manual parsing is useful for understanding the underlying file structure and for learning how simulation data are represented.

However, in real computational workflows, manually parsing every structure file would be repetitive and error-prone.

Scientific libraries such as `pymatgen` provide tested and reusable tools that allow us to focus on the scientific problem rather than repeatedly handling low-level file parsing.

The general workflow is therefore:

**understand the file format manually → use a validated library for routine calculations**

## 4. Write Structure Objects Back to Files
- Structure → POSCAR
- Structure → CIF

In [6]:
from pathlib import Path
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar

In [7]:
# Original POSCAR path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data_root = (
    project_root
    / "data"
    / "raw"
    / "mgo_defect_thermodynamics_dataset"
)

poscar_path = (
    data_root
    / "02_bulk_relaxed"
    / "POSCAR"
)

In [8]:
# Read POSCAR into a pymatgen Structure object
structure = Structure.from_file(poscar_path)

In [9]:
# Define a new output path
new_poscar_path = (
    project_root
    / "data"
    / "processed"
    / "MgO_POSCAR_copy"
)

In [10]:
# Make sure the output folder exists
new_poscar_path.parent.mkdir(parents=True, exist_ok=True)

In [11]:
# Write the Structure object back into POSCAR format
Poscar(structure).write_file(new_poscar_path)

print("New POSCAR written to:")
print(new_poscar_path)

New POSCAR written to:
/Users/yu/Desktop/defect_dynamic/data/processed/MgO_POSCAR_copy


## 5. Summary

In this notebook, I learned the basic relationship between crystal-structure files and Python objects.

The main idea is simple:

**structure file → Python object → inspect or modify → write back to file**

By first parsing a POSCAR manually, I understood how structural information is stored in a text file. I then used `pymatgen` to represent the same information as a `Structure` object and write it back to a new POSCAR.

The key takeaway is that manual parsing helps build understanding, while scientific libraries provide a more reliable and efficient way to handle routine structure operations.